In [1]:
import pandas as pd
import numpy as np
import pandasql as ps

In [ ]:
rewards = pd.read_parquet('../int/rewards_epoch.parquet', engine='auto')
rewards = rewards[['validator_index', 'earnings', 'epoch', 'pool', 'category', 'pool_size_label']]
rewards['earnings'] = rewards['earnings'] / 1e9
rewards['apy'] = rewards['earnings'] / 32 * 100 * 365
rewards_size = rewards.pivot_table(index='epoch', columns='pool_size_label', values='apy', dropna=False, aggfunc='mean')
rewards_category = rewards.pivot_table(index='epoch', columns='category', values='apy', dropna=False, aggfunc='mean')
rewards_pool = rewards.pivot_table(index='epoch', columns='pool', values='apy', dropna=False, aggfunc='mean')
del rewards

In [ ]:
rewards_size = rewards_size.loc[:, rewards_size.columns.notna()]
rewards_category = rewards_category.loc[:, rewards_category.columns.notna()]
rewards_pool = rewards_pool.loc[:, rewards_pool.columns.notna()]

In [ ]:
rewards_size

pool_size_label,1,100+,2-5,20-99,6-19
epoch,,,,,
213619,3.228596,3.523359,3.328382,3.354958,3.360861
213844,3.191046,3.516477,3.327047,3.421507,3.397688
214069,3.165034,3.513273,3.310877,3.441939,3.375904
214294,3.160333,3.498146,3.318798,3.479943,3.481446
214519,3.126423,3.497919,3.337881,3.434020,3.352922
...,...,...,...,...,...
282244,2.669179,2.834160,2.711396,2.806159,2.757794
282469,2.592093,2.847913,2.796677,2.803181,2.772841
282694,2.668939,2.823012,2.693952,2.800848,2.715044


In [ ]:
rewards_size.reset_index(inplace=True)

# Set the initial known value FIX EPOCH
initial_index = rewards_size[rewards_size['epoch'] == 213619].index[0]
rewards_size.loc[initial_index, 'slot'] = 6840000

# Fill subsequent rows
for i in range(initial_index + 1, len(rewards_size)):
    rewards_size.loc[i, 'slot'] = rewards_size.loc[i - 1, 'slot'] + 7200

# Backfill previous rows
for i in range(initial_index - 1, -1, -1):
    rewards_size.loc[i, 'slot'] = rewards_size.loc[i + 1, 'slot'] - 7200

rewards_size.set_index('epoch', inplace=True)

rewards_size

pool_size_label,1,100+,2-5,20-99,6-19,slot
epoch,,,,,,
213619,3.228596,3.523359,3.328382,3.354958,3.360861,6840000.0
213844,3.191046,3.516477,3.327047,3.421507,3.397688,6847200.0
214069,3.165034,3.513273,3.310877,3.441939,3.375904,6854400.0
214294,3.160333,3.498146,3.318798,3.479943,3.481446,6861600.0
214519,3.126423,3.497919,3.337881,3.434020,3.352922,6868800.0
...,...,...,...,...,...,...
282244,2.669179,2.834160,2.711396,2.806159,2.757794,9036000.0
282469,2.592093,2.847913,2.796677,2.803181,2.772841,9043200.0
282694,2.668939,2.823012,2.693952,2.800848,2.715044,9050400.0


In [ ]:
rewards_category.reset_index(inplace=True)

# Set the initial known value FIX EPOCH
initial_index = rewards_category[rewards_category['epoch'] == 213619].index[0]
rewards_category.loc[initial_index, 'slot'] = 6840000

# Fill subsequent rows
for i in range(initial_index + 1, len(rewards_category)):
    rewards_category.loc[i, 'slot'] = rewards_category.loc[i - 1, 'slot'] + 7200

# Backfill previous rows
for i in range(initial_index - 1, -1, -1):
    rewards_category.loc[i, 'slot'] = rewards_category.loc[i + 1, 'slot'] - 7200

rewards_category.set_index('epoch', inplace=True)

rewards_category

category,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,slot
epoch,,,,,,,
213619,3.532871,3.990553,3.516661,3.453131,3.500298,3.458661,6840000.0
213844,3.507957,3.217049,3.534459,3.501663,3.400182,3.469874,6847200.0
214069,3.545429,3.670446,3.506144,3.516955,3.284053,3.476849,6854400.0
214294,3.513261,3.509596,3.501175,3.457467,3.369128,3.478299,6861600.0
214519,3.516890,3.797065,3.507990,3.455326,3.327462,3.450159,6868800.0
...,...,...,...,...,...,...,...
282244,2.842207,2.841924,2.848308,2.735272,2.827816,2.793060,9036000.0
282469,2.842548,2.862795,2.851137,2.771063,2.835482,2.827567,9043200.0
282694,2.815299,2.796924,2.842588,2.880818,2.822105,2.788734,9050400.0


In [ ]:
rewards_pool.reset_index(inplace=True)

# Set the initial known value FIX EPOCH
initial_index = rewards_pool[rewards_pool['epoch'] == 213619].index[0]
rewards_pool.loc[initial_index, 'slot'] = 6840000

# Fill subsequent rows
for i in range(initial_index + 1, len(rewards_pool)):
    rewards_pool.loc[i, 'slot'] = rewards_pool.loc[i - 1, 'slot'] + 7200

# Backfill previous rows
for i in range(initial_index - 1, -1, -1):
    rewards_pool.loc[i, 'slot'] = rewards_pool.loc[i + 1, 'slot'] - 7200

rewards_pool.set_index('epoch', inplace=True)

rewards_pool

pool,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,slot
epoch,,,,,,,,,,,,
213619,3.539049,3.522994,3.545553,NaN,3.540595,3.486309,3.520150,NaN,3.472621,3.474172,3.442935,6840000.0
213844,3.541451,3.450061,3.512527,NaN,3.496393,3.515131,3.534774,NaN,3.496143,3.470035,3.484914,6847200.0
214069,3.578010,3.619082,3.536630,NaN,3.534248,3.429357,3.509834,NaN,3.529406,3.457958,3.456533,6854400.0
214294,3.554113,3.535055,3.490278,NaN,3.585313,3.514633,3.500818,NaN,3.470252,3.459413,3.497260,6861600.0
214519,3.538729,3.450040,3.518947,NaN,3.560985,3.529333,3.510841,NaN,3.465220,3.435049,3.484705,6868800.0
...,...,...,...,...,...,...,...,...,...,...,...,...
282244,2.801437,2.737715,2.876464,2.832276,2.846830,2.858339,2.864946,2.808386,2.659916,2.804539,2.681956,9036000.0
282469,2.873345,2.746686,2.843657,2.813879,2.818294,2.818643,2.862564,2.844126,2.869105,2.836833,2.749377,9043200.0
282694,2.889525,2.634568,2.822819,2.800004,2.769761,2.730961,2.856369,2.813590,2.808366,2.804378,2.703786,9050400.0


In [ ]:
rewards_size['total'] = rewards_size.mean(axis=1)
rewards_category['total'] = rewards_category.mean(axis=1)
rewards_pool['total'] = rewards_pool.mean(axis=1)

In [ ]:
rewards_size.to_csv('../int/rewards_size.csv')
rewards_category.to_csv('../int/rewards_category.csv')
rewards_pool.to_csv('../int/rewards_pool.csv')